# Flux — scFEA metabolni flux (TRIM-Flux Var 2)

Izracuna metabolni flux za vsako celico prek **scFEA** (single-cell Flux Estimation Analysis).
Flux postane 3. modaliteta v TRIM (poleg RNA + TCR).

**Vhod:** `data_rna_counts.pkl` (SUROVI counti — scFEA jih sam normalizira; NE normalizirani `data_rna.pkl`)
**Izhod:** `data_flux.pkl` — matrika (celice x ~168 metabolnih modulov), poravnana z `data_labels`

Orodje scFEA (izbrano po raziskavi izvedljivosti): GNN, GPU, per-cell, dropout-robusten (korelacija >0.85),
~168 cloveskih metabolnih modulov. Nevzdrzevan od 2021 -> potrebni patchi za moderni Colab (spodaj).

> scFEA fluksi so RELATIVNI/model-odvisni (ne absolutne hitrosti); benchmark scFEA/Compass/METAFlux ne obstaja.


## 0. Mount + namestitev scFEA (+ patchi za moderni Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -q https://github.com/changwn/scFEA.git
%cd /content/scFEA

# scFEA importa 'magic' na vrhu skripte -> nujno nalozen (tudi pri sc_imputation=False).
# --no-deps: sicer vlece star pandas iz vira -> build pade.
!pip install -q --no-deps magic-impute graphtools scprep s_gd2 pygsp Deprecated tasklogger wrapt

# PATCHI (scFEA pisan za pandas<2 / stari torch):
!sed -i -E 's/([A-Za-z_]*Df)\.append\(/\1._append(/g' src/scFEA.py   # pandas: .append -> ._append
!sed -i 's/\.detach()\.numpy()/.detach().cpu().numpy()/g' src/scFEA.py  # torch: GPU tensor -> cpu

import magic
print('magic:', getattr(magic, '__version__', 'OK'))
print('cmMat na voljo:')
!ls data/ | grep -iE 'module_gene|cmMat'

## 1. Nalozi surove counte + gene imena

scFEA hoce **surove counte** (sam logira ce max>50) in **gene simbole** (vrstice=geni, stolpci=celice).
`data_rna_counts.pkl` je shranjen v notebooku 01 (loceno od normaliziranega `data_rna.pkl`).

In [ ]:
import pickle, numpy as np, pandas as pd, os, time
from scipy.sparse import issparse

DATA = '/content/drive/MyDrive/Diploma/data/processed'
with open(os.path.join(DATA, 'data_rna_counts.pkl'), 'rb') as f:
    counts = pickle.load(f)                 # SUROVI counti (celice x geni), sparse
with open(os.path.join(DATA, 'gene_names.pkl'), 'rb') as f:
    gene_names = [str(g) for g in pickle.load(f)]

print('counts:', counts.shape, type(counts).__name__)
print('gene_names:', len(gene_names), '| primer:', gene_names[:4])
assert len(gene_names) == counts.shape[1], 'gene_names != stolpci counts!'

# scFEA rabi gene SIMBOLE (CD8A), ne Ensembl ID (ENSG...).
is_ensembl = all(g.upper().startswith('ENSG') for g in gene_names[:50])
assert not is_ensembl, 'Geni so Ensembl ID -> scFEA rabi simbole (pretvori z mygene)!'
print('Geni so simboli:', not is_ensembl)

## 2. Pripravi scFEA vhod (CSV: geni x celice)

Celoten dataset. scFEA transponira interno; mi damo geni=vrstice, celice=stolpci (kot zahteva dokumentacija).

In [ ]:
# KLJUC: scFEA uporabi le ~663 modulnih genov (od 36601). Filtriraj CSV nanje ->
# 55x manjsi CSV -> branje/pisanje v sekundah (sicer polna matrika = 29+ min / crash).
# Rezultat je IDENTICEN (scFEA tako ali tako vzame le presek modulnih genov).
mg = pd.read_csv('/content/scFEA/data/module_gene_m168.csv', index_col=0)
module_genes = set()
for col in mg.columns:
    for v in mg[col].dropna().astype(str):
        g = v.strip()
        if g and g.lower() != 'nan':
            module_genes.add(g)
print('scFEA modulnih genov:', len(module_genes))

# indeksi nasih genov, ki so v modulih (ohrani vrstni red)
keep_idx = [i for i, g in enumerate(gene_names) if g in module_genes]
keep_names = [gene_names[i] for i in keep_idx]
print(f'nasih genov v modulih: {len(keep_idx)} (od {len(gene_names)})')
assert len(keep_idx) > 100, 'premalo ujemanja gene-imen!'

# sparse-ohranjajoce: izberi le te stolpce (gene), sele nato -> gosta (majhna) matrika
Xk = counts[:, keep_idx]
Xk = Xk.toarray() if issparse(Xk) else np.asarray(Xk)
Xk = np.rint(Xk).astype(np.int32)
print('Filtriran X (celice x modulni geni):', Xk.shape, '| min/max:', Xk.min(), Xk.max())

# scFEA CSV: vrstice=geni, stolpci=celice
df_in = pd.DataFrame(Xk.T, index=keep_names, columns=[f'c{i}' for i in range(Xk.shape[0])])
os.makedirs('/content/scfea_input', exist_ok=True)
in_csv = '/content/scfea_input/expr.csv'
df_in.to_csv(in_csv)
sz = os.path.getsize(in_csv) / 1e6
print(f'scFEA vhod (geni x celice): {df_in.shape} -> {in_csv} ({sz:.0f} MB)')
del Xk, df_in; import gc; gc.collect()

## 3. Pozeni scFEA (cel dataset)

Cloveski model: `module_gene_m168.csv` + `cmMat_c70_m168.csv`. Cas ~min (GNN na GPU).

In [ ]:
%cd /content/scFEA
os.makedirs('/content/scfea_output', exist_ok=True)
flux_csv = '/content/scfea_output/flux.csv'

t0 = time.time()
!python src/scFEA.py \
    --data_dir data \
    --input_dir /content/scfea_input \
    --test_file expr.csv \
    --moduleGene_file module_gene_m168.csv \
    --stoichiometry_matrix cmMat_c70_m168.csv \
    --output_flux_file {flux_csv} \
    --output_balance_file /content/scfea_output/balance.csv \
    --sc_imputation False
print(f'\n=== scFEA cas: {time.time()-t0:.0f} s za {counts.shape[0]} celic ===')

## 4. Preveri flux + shrani `data_flux.pkl`

Flux mora biti brez NaN in poravnan z `data_labels` po vrsticah.

In [ ]:
flux = pd.read_csv(flux_csv, index_col=0)
print('Flux surova oblika:', flux.shape)

# scFEA lahko celice postavi v STOLPCE ali VRSTICE, in lahko preuredi vrstni red.
# Poravnaj na NAS vrstni red celic (c0, c1, ... = data_labels vrstni red).
expected = [f'c{i}' for i in range(counts.shape[0])]
if flux.shape[0] == len(expected):           # celice v vrsticah
    pass
elif flux.shape[1] == len(expected):         # celice v stolpcih -> transponiraj
    flux = flux.T
    print('  (flux transponiran: celice so bile v stolpcih)')
else:
    raise AssertionError(f'flux oblika {flux.shape} ne ustreza {len(expected)} celicam!')

# Reindeksiraj na nas vrstni red (varovalo proti preurejanju)
missing = set(expected) - set(flux.index.astype(str))
assert not missing, f'flux manjkajo celice: {list(missing)[:5]}...'
flux = flux.loc[expected]                    # EKSPLICITNA poravnava po imenu
print('Flux poravnan:', flux.shape, '(celice x moduli, v data_labels vrstnem redu)')

n_nan = int(np.isnan(flux.values).sum())
print('NaN:', n_nan, '| delez nicelnih:', f'{(flux.values==0).mean():.3f}',
      '| min/max/mean:', f'{flux.values.min():.3f}/{flux.values.max():.3f}/{flux.values.mean():.3f}')
assert n_nan == 0, 'FLUX VSEBUJE NaN -> preveri vhod (surovi counti? gene ujemanje?)'

# Shrani kot numpy (poravnan z data_labels po vrsticah, kot data_rna/data_tcr)
data_flux = flux.values.astype(np.float32)
with open(os.path.join(DATA, 'data_flux.pkl'), 'wb') as f:
    pickle.dump(data_flux, f)
# shrani tudi imena modulov (za interpretacijo)
with open(os.path.join(DATA, 'flux_module_names.pkl'), 'wb') as f:
    pickle.dump(list(flux.columns), f)
print('\ndata_flux.pkl shranjen:', data_flux.shape, '(3. modaliteta za TRIM)')

## 5. Zakljucek

- `data_flux.pkl` (celice x ~168 modulov) = metabolni flux, poravnan z RNA/TCR.
- Naslednje: flux encoder/decoder kot 3. modaliteta v TRIM (RNA + TCR + Flux, Var 2).

Pridrzki za diplomo: scFEA fluksi RELATIVNI/model-odvisni; scFEA nevzdrzevan (patchi za pandas/torch).